# Experimental cached-mask M2 benchmark: Colab Free vs Pro

Run two independent Colab runtimes with manually selected FREE/PRO labels and identical COG, polygons, subset and settings. Run directories are unique, but runtimes share GCS quota and network bandwidth. No EE task, GCS write, overwrite or delete is performed. Billing, permissions and runtime limits apply.

This is a **prototype**, not production M2. A polygon mask is rasterized once on the source grid and reused by 1/2/4 window workers, subject to CPU/RAM admission control. Workers own separate read handles and a single main-thread writer coordinates output. Subset rasterization uses rasterio in RAM (maximum 64 million pixels); national mode requires `gdal_rasterize` and avoids a national array in RAM. Every repetition produces a BigTIFF COG and is validated over all pixels against original data, validity mask and a separately rasterized reference. The reference uses related GDAL rasterization semantics and cannot prove the supplied geometry is correct. Zero is not assumed nodata.

Application includes local reads, mask operations, writes and LZW compression; COG conversion is measured separately. Worker order rotates by round and is recorded, but OS/GDAL caches and prior stages still bias timings. Only application median is projected to national pixels: this is **VERY UNCERTAIN**, not a promise. Download, rasterization, COG and validation are never extrapolated. National times are actual when explicitly opted in. RSS/CPU cover only the parent process; RSS sampling can miss peaks. Controls and estimates are not hard limits. Retained intermediates can occupy substantial disk.

GDAL receives a fixed token refreshed once before download; it can expire during long national runs. The national reference repeats the same `gdal_rasterize` method, not an independent correctness check. A validity mask does **not** protect pixel confidentiality: values outside the polygon remain in the files. Reports can contain private URIs, paths and errors; inspect them before downloading or sharing.

The module defaults to a fixed published commit for reproducibility; change it only to another trusted revision or point to a local checkout. Optionally supply an existing Earth Engine FeatureCollection for read-only geometry (`getInfo`, no export). Use a representative boundary subset and identical inputs. Offline regression: `python network/catalog/development/version_04/tests/m2_mask_benchmark/benchmark_m2.py`.

In [ ]:
# @title Configuration — identical in both runtimes except label
CONFIG = {
    "label": "FREE",  # Opções: FREE ou PRO; escolha conforme a sessão
    "gcs_uri": "",  # PREENCHA com COG existente; exemplo: gs://meu-bucket/pasta/imagem.tif
    "geometry_path": "/content/country.geojson",  # Se local, envie GeoJSON e ajuste; exemplo: /content/pais.geojson
    "geometry_crs": "EPSG:4326",  # CRS real do GeoJSON; exemplos: EPSG:4326 ou EPSG:32718
    "mode": "subset", "confirm_national": False,  # mode: subset ou national; nacional também exige confirm_national=True
    "subset_col": 0, "subset_row": 0, "subset_width": 1024, "subset_height": 1024,  # Pixels: origem (coluna, linha) e tamanho; exemplo: 1000, 2000, 4096, 4096
    "window_size": 512, "window_workers": [1, 2, 4], "repeats": 2,  # Tamanho: múltiplo de 16 entre 16 e 4096; workers: lista sem repetição de 1, 2, 4; repeats: 1 a 10
    "threads": 1, "gdal_cache_mb": 128, "disk_reserve_gb": 5,  # Exemplos: 1 thread, cache 128 MB, reserva 5 GB; mantenha iguais para comparar
    "work_root": "/content/m2_benchmark_runs",  # Pode manter; cada execução cria pasta própria
}
LOCAL_MODULE_PATH = ""  # Vazio baixa do GitHub; alternativa: caminho local, ex. /content/benchmark_m2.py
GITHUB_REPO = "mapbiomas/brazil-fire"  # Já preenchido; não precisa alterar
GITHUB_COMMIT = "338b79ebe62a43cf1cb54732b4a9e27db085fd91"  # Já preenchido; versão fixa do módulo
GEOMETRY_SOURCE = "local"  # Opções: local (GeoJSON enviado) ou ee (FeatureCollection existente)
EE_ASSET = ""  # Se ee, PREENCHA; exemplo: projects/meu-projeto/assets/regioes
EE_PROJECT = ""  # Se ee, PREENCHA; exemplo: meu-projeto-earth-engine
PERSIST_REPORTS_TO_DRIVE = False  # Opções: False não salva; True copia apenas relatórios ao Drive


In [ ]:
# @title Install dependencies, load pinned module, authenticate and load geometry
import sys, subprocess, importlib.util, tempfile, pathlib, re, urllib.request, json
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio", "numpy", "psutil", "google-auth"])
from google.colab import auth
auth.authenticate_user()
if LOCAL_MODULE_PATH:
    module_path = pathlib.Path(LOCAL_MODULE_PATH)
else:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPO) or not re.fullmatch(r"[0-9a-fA-F]{40}", GITHUB_COMMIT):
        raise ValueError("Set trusted GITHUB_REPO and full commit SHA, or LOCAL_MODULE_PATH")
    url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_COMMIT}/network/catalog/development/version_04/tests/m2_mask_benchmark/benchmark_m2.py"
    module_path = pathlib.Path(tempfile.mkdtemp(prefix="m2_module_")) / "benchmark_m2.py"
    with urllib.request.urlopen(url) as response:
        module_path.write_bytes(response.read())
    print("Module source:", url)
spec = importlib.util.spec_from_file_location("benchmark_m2", str(module_path))
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
if GEOMETRY_SOURCE == "ee":
    if not EE_ASSET or not EE_PROJECT:
        raise ValueError("Set EE_ASSET and EE_PROJECT")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "earthengine-api"])
    import ee
    ee.Initialize(project=EE_PROJECT)
    fc = ee.FeatureCollection(EE_ASSET).getInfo()  # read only; no task
    polygons = list(benchmark.polygons(fc))
    if not polygons:
        raise ValueError("FeatureCollection contains no polygons")
    geometry_path = pathlib.Path(tempfile.mkdtemp(prefix="m2_geometry_")) / "polygons.geojson"
    geometry_path.write_text(json.dumps({"type": "GeometryCollection", "geometries": polygons}), encoding="utf-8")
    CONFIG["geometry_path"] = str(geometry_path)
    CONFIG["geometry_crs"] = "EPSG:4326"
elif GEOMETRY_SOURCE != "local":
    raise ValueError("GEOMETRY_SOURCE must be local or ee")
print("Authenticated; module ready. No remote writes.")


In [ ]:
# @title Run benchmark — each invocation creates a new directory
report, run_dir = None, None  # clear previous results even if this run fails
report, run_dir = benchmark.run(CONFIG)
print("Status:", report["status"], "run:", report["run_id"])
for scenario in report["scenarios"]:
    print(scenario)
for row in report["stages"]:
    print(row)


## Results and persistence

`report.json` contains configuration, repeated application timings, medians and the **very uncertain application-only** projection; `stages.csv` records each stage, including one-time mask creation, COG and validation. Every COG is checked across all local pixels and masks against source and reference. `threads` controls COG/GDAL separately from window workers. Skipped scenarios include a reason. For national mode install `gdal-bin`, set mode and explicit confirmation. National operations may exceed Colab resources.

Download reports below. Drive persistence is **off** unless explicitly enabled; it copies only reports into a new run-ID directory. Raster copies stay local and disappear when the runtime is recycled. Failed runs print their directory and retain partial reports. Avoid sharing reports containing sensitive paths.

In [ ]:
# @title Download reports and optionally persist them to Drive
if report is None or run_dir is None or report.get("status") != "ok":
    raise RuntimeError("No successful current run to download; inspect the printed failed-run directory manually")
from google.colab import files
for name in ("report.json", "stages.csv"):
    files.download(str(run_dir / name))
if PERSIST_REPORTS_TO_DRIVE:
    from google.colab import drive
    from pathlib import Path
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/m2_benchmark_reports") / report["run_id"]
    destination.mkdir(parents=True, exist_ok=False)
    for name in ("report.json", "stages.csv"):
        shutil.copyfile(run_dir / name, destination / name)
    print("Reports saved to", destination)


## Optional: local mosaic of existing GCS chunks

This is a **separate, opt-in, potentially expensive** experiment, not part of the subset projection. Set a specific object prefix (not just a bucket) containing compatible TIFF chunks for one band/period. The Cloud Storage API lists at most `max_chunks + 1` objects, counting non-TIFFs too; exceeding the limit aborts before download. A specific prefix can still match unrelated objects; inspect it carefully. It downloads matching TIFFs, then builds a local VRT and local BigTIFF COG; no GEE export or remote writes occur. Install `gdalbuildvrt` if absent (e.g. `apt-get install gdal-bin`); dependencies may vary. Read access is required. Nine sampled windows compare against the VRT, not the source chunks independently. **Agreement with VRT does not prove geographic mask correctness**: chunks may have missing or incorrect nodata/masks, and overlap ordering follows sorted object names and GDAL behavior. This operation does not apply the country polygons. Disk estimate is approximate; sparse mosaics may exceed it. CPU/RSS exclude the `gdalbuildvrt` child process. Reports contain potentially private URIs and errors. After a successful chunk run, select its reports with `run_dir, report = chunks_dir, chunks_report` and rerun the reports cell.

In [ ]:
# @title Optional chunk mosaic (off by default)
RUN_CHUNKS = False  # Opções: False desativa; True ativa após preencher prefix e confirmar download
CHUNKS = {
    "label": CONFIG["label"], "work_root": CONFIG["work_root"],
    "prefix": "",  # Se ativar, PREENCHA; exemplo: gs://meu-bucket/pasta/CHUNKS/banda
    "confirm_chunks": False,  # Opções: False cancela; True autoriza download completo dos chunks
    "max_chunks": 100,  # Limite positivo de objetos, incluindo não TIFFs; exemplo: 300 para 255 chunks
    "threads": CONFIG["threads"],
    "disk_reserve_gb": CONFIG["disk_reserve_gb"],
}
chunks_report, chunks_dir = None, None
if RUN_CHUNKS:
    report, run_dir = None, None  # invalidate selected reports before attempting a new run
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-cloud-storage"])
    chunks_report, chunks_dir = benchmark.run_chunks(CHUNKS)
    print(chunks_report)
